In [1]:
! pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 MB 109.3 MB/s  0:00:0200:0100:01


In [2]:
! pip install transformers

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import itertools
from flash_attn import flash_attn_varlen_func, flash_attn_with_kvcache

class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale
        if self.shift is not None:
            norm_x = norm_x + self.shift
        return norm_x.to(input_dtype)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False, dtype=cfg["dtype"])
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False, dtype=cfg["dtype"])
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False, dtype=cfg["dtype"])

    def forward(self, x):
        return self.fc3(F.silu(self.fc1(x)) * self.fc2(x))

def apply_rope(x, cos, sin):
    # RoPE 
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)


class GroupedQueryAttention(nn.Module):
    def __init__(self, layer_idx, cfg):
        super().__init__()
        self.layer_idx = layer_idx
        self.num_heads = cfg["n_heads"]
        self.num_kv_groups = cfg["n_kv_groups"]
        self.head_dim = cfg["head_dim"]

        self.W_query = nn.Linear(cfg["emb_dim"], self.num_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.W_key = nn.Linear(cfg["emb_dim"],   self.num_kv_groups * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.W_value = nn.Linear(cfg["emb_dim"],  self.num_kv_groups * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.out_proj = nn.Linear(self.num_heads * self.head_dim, cfg["emb_dim"], bias=False, dtype=cfg["dtype"])
        
        self.q_norm = RMSNorm(self.head_dim) if cfg.get("qk_norm") else None
        self.k_norm = RMSNorm(self.head_dim) if cfg.get("qk_norm") else None

    def forward(self, x, k_cache, v_cache, metadata):
        q = self.W_query(x).view(-1, self.num_heads, self.head_dim)
        k = self.W_key(x).view(-1, self.num_kv_groups, self.head_dim)
        v = self.W_value(x).view(-1, self.num_kv_groups, self.head_dim)
        
        if self.q_norm:
            q, k = self.q_norm(q), self.k_norm(k)

        q = apply_rope(q, metadata['cos'], metadata['sin'])
        k = apply_rope(k, metadata['cos'], metadata['sin'])

        # Update Paged Cache
        slots = metadata['slot_mapping']
        b_idx = slots // metadata['block_size']
        o_idx = slots % metadata['block_size']
        
      
        # shape = (max_blocks, self.block_size, cfg["n_kv_groups"], cfg["head_dim"])
        k_cache[b_idx, o_idx] = k
        v_cache[b_idx, o_idx] = v

        if metadata['is_decoding']:
            attn_out = flash_attn_with_kvcache(
                q.unsqueeze(1), k_cache, v_cache,
                cache_seqlens=metadata['seqlens'],
                block_table=metadata['block_table'],
                causal=True
            )
        else:
            attn_out = flash_attn_varlen_func(
                q, k, v,
                cu_seqlens_q=metadata['cu_seqlens'],
                cu_seqlens_k=metadata['cu_seqlens'],
                max_seqlen_q=metadata['max_seqlen'],
                max_seqlen_k=metadata['max_seqlen'],
                causal=True
            )
        
        return self.out_proj(attn_out.view(-1, self.num_heads * self.head_dim))

class TransformerBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.att = GroupedQueryAttention(layer_idx, cfg)
        self.ff = FeedForward(cfg)
        self.norm1, self.norm2 = RMSNorm(cfg["emb_dim"]), RMSNorm(cfg["emb_dim"])

    def forward(self, x, k_cache, v_cache, metadata):
        x = x + self.att(self.norm1(x), k_cache, v_cache, metadata)
        x = x + self.ff(self.norm2(x))
        return x

class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        inv_freq = 1.0 / (cfg["rope_base"] ** (torch.arange(0, cfg["head_dim"], 2).float() / cfg["head_dim"]))
        t = torch.arange(cfg["context_length"]).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        self.register_buffer("cos_buf", freqs.cos().to(cfg["dtype"]))
        self.register_buffer("sin_buf", freqs.sin().to(cfg["dtype"]))

    def forward(self, input_ids, k_caches, v_caches, metadata):
        """
        k_caches and v_caches are lists of tensors (one per layer).
        """
        x = self.tok_emb(input_ids)
        for i, block in enumerate(self.trf_blocks):
            x = block(x, k_caches[i], v_caches[i], metadata)
        return self.out_head(self.final_norm(x))


# block_table → which memory blocks belong to this sequence
# slot_mapping → exact KV cache positions
# SequenceState → runtime state of a request

# For ex:
# block_size = 4
# max_blocks = 8

# Global KV cache memory:

# Block 0 : slots 0 1 2 3
# Block 1 : slots 4 5 6 7 
# so on 

class SequenceState:
    def __init__(self, sid, prompt_ids, max_gen_len, cfg, device):
        self.id, self.device, self.block_size = sid, device, cfg["block_size"]
        self.prefill_pos = 0
        p_len = len(prompt_ids)
        self.max_total_len = p_len + max_gen_len
        self.tokens = torch.zeros(self.max_total_len, dtype=torch.long, device=device)
        self.tokens[:p_len] = torch.tensor(prompt_ids, dtype=torch.long, device=device)
        self.num_tokens, self.is_prefill = p_len, True
        
        max_blocks = (self.max_total_len + self.block_size - 1) // self.block_size
        self.block_table = torch.zeros(max_blocks, dtype=torch.int32, device=device) #which memory blocks store KV cache for this sequence  for ex B1,B2,B3 or B2,B4,B7
        self.slot_mapping = torch.zeros(self.max_total_len, dtype=torch.long, device=device) #index inside the blocks for each token kv cache for ex [0,1,2] in B1 if block_size = 3
        self.block_count = 0

    def update_metadata(self):
        start = 0 if self.is_prefill else self.num_tokens - 1
        idx = torch.arange(start, self.num_tokens, device=self.device)
        
        b_idx, o_idx = idx // self.block_size, idx % self.block_size
        #token position → (block index and block position)
        #token 0 → block 0 offset 0
        #token 1 → block 0 offset 1

        
        self.slot_mapping[idx] = self.block_table[b_idx].long() * self.block_size + o_idx
        # where each token KV cache is and During decode it fetches keys like: k_cache[slot]

class PagedKVManager(nn.Module):
    def __init__(self, cfg, max_blocks, device):
        super().__init__()
        self.block_size = cfg["block_size"]
        self.free_blocks = deque(range(max_blocks))
        shape = (max_blocks, self.block_size, cfg["n_kv_groups"], cfg["head_dim"])
        
        self.k_cache = nn.ParameterList([
            nn.Parameter(torch.zeros(shape, device=device, dtype=cfg["dtype"]), requires_grad=False) 
            for _ in range(cfg["n_layers"])
        ])
        self.v_cache = nn.ParameterList([
            nn.Parameter(torch.zeros(shape, device=device, dtype=cfg["dtype"]), requires_grad=False) 
            for _ in range(cfg["n_layers"])
        ])
    
    def can_allocate(self, num_tokens):
        """Check if we have enough blocks for the initial prefill."""
        needed = (num_tokens + self.block_size - 1) // self.block_size
        return len(self.free_blocks) >= needed
    
    def allocate(self, state):

        needed = (state.num_tokens + self.block_size - 1) // self.block_size
        while state.block_count < needed:
            if not self.free_blocks: 
                raise RuntimeError("KV Cache full! (Engine should have prevented this)")
            state.block_table[state.block_count] = self.free_blocks.popleft()
            state.block_count += 1


    def free(self, state):
        for i in range(state.block_count):
            self.free_blocks.append(int(state.block_table[i]))
        state.block_count = 0


        
class ContinuousBatchEngine:
    def __init__(self, model, kv_mgr, cfg):
        
       
        self.model = model
        self.model.eval()
        
        self.kv_mgr = kv_mgr
        self.cfg = cfg
        self.active = {}
        self.id_gen = itertools.count()
        self.device = next(model.parameters()).device
        self.max_batch = cfg.get("max_batch_size", 16)
        self.waiting_room = deque() # Queue for sequences waiting for KV blocks
        
        # Buffers
        self.cu_seqlens_buf = torch.zeros(self.max_batch + 1, dtype=torch.int32, device=self.device)
        self.seqlens_buf = torch.zeros(self.max_batch, dtype=torch.int32, device=self.device)
        self.arange_buf = torch.arange(cfg["context_length"], device=self.device)
    
    def add_sequence(self, prompt_ids, max_gen_len=128):
        """Adds to waiting room instead of active immediately."""
        sid = next(self.id_gen)
        # Store metadata needed to initialize State later
        self.waiting_room.append({
            'sid': sid, 
            'prompt_ids': prompt_ids, 
            'max_gen_len': max_gen_len
        })
    
    def _try_schedule_waiting(self):
        """Moves sequences from waiting room to active if memory permits."""
        while self.waiting_room and len(self.active) < self.max_batch:
            next_req = self.waiting_room[0] # Peek
            p_len = len(next_req['prompt_ids'])
            
            # check if KV manager has room for the prefill
            if self.kv_mgr.can_allocate(p_len):
                req = self.waiting_room.popleft()
                self.active[req['sid']] = SequenceState(
                    req['sid'], req['prompt_ids'], req['max_gen_len'], self.cfg, self.device
                )
            else:
                # Cache is full, stop trying to add more for this step
                break


    def step(self):
        self._try_schedule_waiting()
        if not self.active: return {}
        states = list(self.active.values())
        
        for s in states:
            self.kv_mgr.allocate(s)
            s.update_metadata()

        metadata, input_ids = self._prepare_inference_data(states)

        #cache are already in list do we need to pass it as list?
        with torch.no_grad():
            logits = self.model(
                input_ids, 
                list(self.kv_mgr.k_cache), 
                list(self.kv_mgr.v_cache), 
                metadata
            )

        # Sampling and State Update
        last_token_indices = metadata['cu_seqlens'][1:] - 1
        next_tokens = torch.argmax(logits[last_token_indices], dim=-1)

        finished = {}
        for i, (s, next_token) in enumerate(zip(states, next_tokens)):
            token_id = next_token.item()
            s.is_prefill = False
            s.tokens[s.num_tokens] = token_id
            s.num_tokens += 1
            if token_id in [151643, 151645] or s.num_tokens >= s.max_total_len:
                finished[s.id] = s.tokens[:s.num_tokens].tolist()
                self.kv_mgr.free(s)
                del self.active[s.id]
        return finished

    def _prepare_inference_data(self, states):
        num_seqs = len(states)
        is_decoding = all(not s.is_prefill for s in states)
        
        slots_to_cat = []
        pos_to_cat = []
        curr_cu = 0
        self.cu_seqlens_buf[0] = 0
        
        for i, s in enumerate(states):
            self.seqlens_buf[i] = s.num_tokens
            length = s.num_tokens if s.is_prefill else 1
            start = 0 if s.is_prefill else s.num_tokens - 1
            
            slots_to_cat.append(s.slot_mapping[start:s.num_tokens])
            pos_to_cat.append(self.arange_buf[start:s.num_tokens])
            
            curr_cu += length
            self.cu_seqlens_buf[i+1] = curr_cu

        flat_pos = torch.cat(pos_to_cat)
        
        # Build metadata dict (ensure all values are Tensors or simple bool/ints)
        metadata = {
            'is_decoding': is_decoding,
            'seqlens': self.seqlens_buf[:num_seqs],
            'slot_mapping': torch.cat(slots_to_cat),
            'cu_seqlens': self.cu_seqlens_buf[:num_seqs + 1],
            'max_seqlen': int(max(s.num_tokens if s.is_prefill else 1 for s in states)),
            'block_table': torch.stack([s.block_table for s in states]),
            'cos': self.model.cos_buf[flat_pos].unsqueeze(1),
            'sin': self.model.sin_buf[flat_pos].unsqueeze(1),                #self.model._orig_mod.cos_buf[flat_pos].unsqueeze(1)
            'block_size': self.cfg["block_size"]
        }
        
        input_ids = torch.cat([
            s.tokens[:s.num_tokens] if s.is_prefill else s.tokens[s.num_tokens-1:s.num_tokens] 
            for s in states
        ])
        
        return metadata, input_ids

**Prefix caching**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import itertools
from flash_attn import flash_attn_varlen_func, flash_attn_with_kvcache


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale
        if self.shift is not None:
            norm_x = norm_x + self.shift
        return norm_x.to(input_dtype)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False, dtype=cfg["dtype"])
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False, dtype=cfg["dtype"])
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False, dtype=cfg["dtype"])

    def forward(self, x):
        return self.fc3(F.silu(self.fc1(x)) * self.fc2(x))

def apply_rope(x, cos, sin):
    # RoPE
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    return torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)


class GroupedQueryAttention(nn.Module):
    def __init__(self, layer_idx, cfg):
        super().__init__()
        self.layer_idx = layer_idx
        self.num_heads = cfg["n_heads"]
        self.num_kv_groups = cfg["n_kv_groups"]
        self.head_dim = cfg["head_dim"]

        self.W_query = nn.Linear(cfg["emb_dim"], self.num_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.W_key = nn.Linear(cfg["emb_dim"],   self.num_kv_groups * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.W_value = nn.Linear(cfg["emb_dim"],  self.num_kv_groups * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.out_proj = nn.Linear(self.num_heads * self.head_dim, cfg["emb_dim"], bias=False, dtype=cfg["dtype"])
        
        self.q_norm = RMSNorm(self.head_dim) if cfg.get("qk_norm") else None
        self.k_norm = RMSNorm(self.head_dim) if cfg.get("qk_norm") else None

    def forward(self, x, k_cache, v_cache, metadata):
        q = self.W_query(x).view(-1, self.num_heads, self.head_dim)
        k = self.W_key(x).view(-1, self.num_kv_groups, self.head_dim)
        v = self.W_value(x).view(-1, self.num_kv_groups, self.head_dim)
        
        if self.q_norm:
            q, k = self.q_norm(q), self.k_norm(k)

        q = apply_rope(q, metadata['cos'], metadata['sin'])
        k = apply_rope(k, metadata['cos'], metadata['sin'])

        # Update Paged Cache
        slots = metadata['slot_mapping']
        b_idx = slots // metadata['block_size']
        o_idx = slots % metadata['block_size']
        
      
        # shape = (max_blocks, self.block_size, cfg["n_kv_groups"], cfg["head_dim"])
        k_cache[b_idx, o_idx] = k
        v_cache[b_idx, o_idx] = v

        if metadata['is_decoding']:
            attn_out = flash_attn_with_kvcache(
                q.unsqueeze(1), k_cache, v_cache,
                cache_seqlens=metadata['seqlens'],
                block_table=metadata['block_table'],
                causal=True
            )
        else:
            attn_out = flash_attn_varlen_func(
                q, k, v,
                cu_seqlens_q=metadata['cu_seqlens'],
                cu_seqlens_k=metadata['cu_seqlens'],
                max_seqlen_q=metadata['max_seqlen'],
                max_seqlen_k=metadata['max_seqlen'],
                causal=True
            )
        
        return self.out_proj(attn_out.view(-1, self.num_heads * self.head_dim))

class TransformerBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.att = GroupedQueryAttention(layer_idx, cfg)
        self.ff = FeedForward(cfg)
        self.norm1, self.norm2 = RMSNorm(cfg["emb_dim"]), RMSNorm(cfg["emb_dim"])

    def forward(self, x, k_cache, v_cache, metadata):
        x = x + self.att(self.norm1(x), k_cache, v_cache, metadata)
        x = x + self.ff(self.norm2(x))
        return x

class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        inv_freq = 1.0 / (cfg["rope_base"] ** (torch.arange(0, cfg["head_dim"], 2).float() / cfg["head_dim"]))
        t = torch.arange(cfg["context_length"]).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        self.register_buffer("cos_buf", freqs.cos().to(cfg["dtype"]))
        self.register_buffer("sin_buf", freqs.sin().to(cfg["dtype"]))

    def forward(self, input_ids, k_caches, v_caches, metadata):
        """
        k_caches and v_caches are lists of tensors (one per layer).
        Passing them as a list of tensors is torch.compile friendly.
        """
        x = self.tok_emb(input_ids)
        for i, block in enumerate(self.trf_blocks):
            x = block(x, k_caches[i], v_caches[i], metadata)
        return self.out_head(self.final_norm(x))

class RadixNode:
    def __init__(self, block_id):
        self.block_id = block_id
        self.children = {}  # Map: Tuple of tokens -> RadixNode
        self.ref_count = 0

class SequenceState:
    def __init__(self, sid, prompt_ids, max_gen_len, cfg, device, matched_blocks=None):
        self.id, self.device, self.block_size = sid, device, cfg["block_size"]
        
        # How many tokens are already in the cache?
        self.prefix_len = len(matched_blocks) * self.block_size if matched_blocks else 0
        p_len = len(prompt_ids)
        
        self.max_total_len = p_len + max_gen_len
        self.tokens = torch.zeros(self.max_total_len, dtype=torch.long, device=device)
        self.tokens[:p_len] = torch.tensor(prompt_ids, dtype=torch.long, device=device)
        
        self.num_tokens = p_len
        self.is_prefill = True 
        
        max_blocks = (self.max_total_len + self.block_size - 1) // self.block_size
        self.block_table = torch.zeros(max_blocks, dtype=torch.int32, device=device)
        
        # Track which blocks are already matched
        self.block_count = 0
        if matched_blocks:
            for b_id in matched_blocks:
                self.block_table[self.block_count] = b_id
                self.block_count += 1

        self.slot_mapping = torch.zeros(self.max_total_len, dtype=torch.long, device=device)

    def update_metadata(self):
        # We only update slots for tokens we are CURRENTLY processing
        start = self.prefix_len if self.is_prefill else self.num_tokens - 1
        idx = torch.arange(start, self.num_tokens, device=self.device)
        
        b_idx, o_idx = idx // self.block_size, idx % self.block_size
        self.slot_mapping[idx] = self.block_table[b_idx].long() * self.block_size + o_idx


class PagedKVManager(nn.Module):
    def __init__(self, cfg, max_blocks, device):
        super().__init__()
        self.block_size = cfg["block_size"]
        self.free_blocks = deque(range(max_blocks))
        self.evictable_blocks = deque() # Blocks with ref_count == 0
        
        self.radix_root = RadixNode(block_id=-1)
        self.block_to_node = {} # Reverse mapping for eviction
        
        shape = (max_blocks, self.block_size, cfg["n_kv_groups"], cfg["head_dim"])
        self.k_cache = nn.ParameterList([nn.Parameter(torch.zeros(shape, device=device, dtype=cfg["dtype"]), requires_grad=False) for _ in range(cfg["n_layers"])])
        self.v_cache = nn.ParameterList([nn.Parameter(torch.zeros(shape, device=device, dtype=cfg["dtype"]), requires_grad=False) for _ in range(cfg["n_layers"])])

    def get_prefix_blocks(self, token_ids):
        """Find matches in Radix Tree and increment their ref counts."""
        matched_blocks = []
        current = self.radix_root
        num_potential_blocks = len(token_ids) // self.block_size
        
        for i in range(num_potential_blocks):
            prefix = tuple(token_ids[i * self.block_size : (i + 1) * self.block_size])
            if prefix in current.children:
                current = current.children[prefix]
                b_id = current.block_id
                matched_blocks.append(b_id)
                # If it was in evictable list, remove it
                if current.ref_count == 0 and b_id in self.evictable_blocks:
                    self.evictable_blocks.remove(b_id)
                current.ref_count += 1
            else:
                break
        return matched_blocks

    def allocate(self, state):
        """Allocates new blocks and registers them in the Radix Tree if full."""
        needed = (state.num_tokens + self.block_size - 1) // self.block_size
        
        while state.block_count < needed:
            if not self.free_blocks:
                if not self.evictable_blocks: raise RuntimeError("KV Cache full!")
                # Evict LRU block
                evict_id = self.evictable_blocks.popleft()
                self._remove_from_radix(evict_id)
                self.free_blocks.append(evict_id)
            
            new_block = self.free_blocks.popleft()
            state.block_table[state.block_count] = new_block
            
            # Register in Radix Tree if it's a complete block
            start_idx = state.block_count * self.block_size
            if state.num_tokens >= start_idx + self.block_size:
                self._register_block(state, state.block_count, new_block)
            
            state.block_count += 1

    def _register_block(self, state, b_idx, b_id):
        # Traverse to parent node and add child
        current = self.radix_root
        for i in range(b_idx):
            p_tokens = tuple(state.tokens[i*self.block_size : (i+1)*self.block_size].tolist())
            current = current.children[p_tokens]
        
        block_tokens = tuple(state.tokens[b_idx*self.block_size : (b_idx+1)*self.block_size].tolist())
        new_node = RadixNode(b_id)
        new_node.ref_count = 1
        current.children[block_tokens] = new_node
        self.block_to_node[b_id] = (current, block_tokens, new_node)

    def _remove_from_radix(self, b_id):
        parent, tokens, node = self.block_to_node.pop(b_id)
        del parent.children[tokens]

    def free(self, state):
        """Decrement ref counts. If 0, move to evictable queue instead of free pool."""
        for i in range(state.block_count):
            b_id = int(state.block_table[i])
            if b_id in self.block_to_node:
                _, _, node = self.block_to_node[b_id]
                node.ref_count -= 1
                if node.ref_count == 0:
                    self.evictable_blocks.append(b_id)
            else:
                # Block was never registered in Radix (partial block), return to free
                self.free_blocks.append(b_id)

class ContinuousBatchEngine:
    def __init__(self, model, kv_mgr, cfg):
        self.model = model
        self.model.eval()
        self.kv_mgr = kv_mgr
        self.cfg = cfg
        self.active = {}
        self.id_gen = itertools.count()
        self.device = next(model.parameters()).device
        self.max_batch = cfg.get("max_batch_size", 16)
        self.waiting_room = deque()

        # Buffers for FlashAttention
        self.cu_seqlens_buf = torch.zeros(self.max_batch + 1, dtype=torch.int32, device=self.device)
        self.seqlens_buf = torch.zeros(self.max_batch, dtype=torch.int32, device=self.device)
        self.arange_buf = torch.arange(cfg["context_length"], device=self.device)

    def add_sequence(self, prompt_ids, max_gen_len=128):
        sid = next(self.id_gen)
        self.waiting_room.append({
            'sid': sid,
            'prompt_ids': prompt_ids,
            'max_gen_len': max_gen_len
        })

    def _try_schedule_waiting(self):
        """Move sequences from waiting room to active if KV cache has room."""
        while self.waiting_room and len(self.active) < self.max_batch:
            req = self.waiting_room[0]
            p_ids = req['prompt_ids']

            # Check prefix cache
            matched_blocks = self.kv_mgr.get_prefix_blocks(p_ids)
            prefix_len = len(matched_blocks) * self.cfg["block_size"]
            total_needed_blocks = (len(p_ids) + self.cfg["block_size"] - 1) // self.cfg["block_size"]
            new_needed = total_needed_blocks - len(matched_blocks)

            if len(self.kv_mgr.free_blocks) >= new_needed:
                # Allocate sequence state with prefix blocks
                self.waiting_room.popleft()
                self.active[req['sid']] = SequenceState(
                    req['sid'], req['prompt_ids'], req['max_gen_len'],
                    self.cfg, self.device, matched_blocks=matched_blocks
                )
            else:
                break  # Not enough KV memory to schedule this sequence

    def step(self):
        self._try_schedule_waiting()
        if not self.active:
            return {}

        states = list(self.active.values())

        # Allocate new blocks for sequences and update slot mappings
        for s in states:
            self.kv_mgr.allocate(s)
            s.update_metadata()

        # Prepare input IDs and metadata
        metadata, input_ids = self._prepare_inference_data(states)

        # Run inference
        with torch.no_grad():
            logits = self.model(input_ids, list(self.kv_mgr.k_cache), list(self.kv_mgr.v_cache), metadata)

        # Sampling / next token
        last_token_indices = metadata['cu_seqlens'][1:] - 1
        next_tokens = torch.argmax(logits[last_token_indices], dim=-1)

        finished = {}
        for i, (s, next_token) in enumerate(zip(states, next_tokens)):
            token_id = next_token.item()

            # Update sequence
            s.is_prefill = False
            s.tokens[s.num_tokens] = token_id
            s.num_tokens += 1

            # Check termination
            if token_id in [151643, 151645] or s.num_tokens >= s.max_total_len:
                finished[s.id] = s.tokens[:s.num_tokens].tolist()
                self.kv_mgr.free(s)
                del self.active[s.id]

        return finished

    def _prepare_inference_data(self, states):
        num_seqs = len(states)
        is_decoding = all(not s.is_prefill for s in states)
        slots_to_cat, pos_to_cat, ids_to_cat = [], [], []
        curr_cu = 0
        self.cu_seqlens_buf[0] = 0

        for i, s in enumerate(states):
            self.seqlens_buf[i] = s.num_tokens
            start = s.prefix_len if s.is_prefill else s.num_tokens - 1
            end = s.num_tokens

            slots_to_cat.append(s.slot_mapping[start:end])
            pos_to_cat.append(self.arange_buf[start:end])
            ids_to_cat.append(s.tokens[start:end])

            curr_cu += end - start
            self.cu_seqlens_buf[i+1] = curr_cu

        flat_pos = torch.cat(pos_to_cat)
        metadata = {
            'is_decoding': is_decoding,
            'seqlens': self.seqlens_buf[:num_seqs],
            'slot_mapping': torch.cat(slots_to_cat),
            'cu_seqlens': self.cu_seqlens_buf[:num_seqs + 1],
            'max_seqlen': int(max((s.num_tokens - s.prefix_len) if s.is_prefill else 1 for s in states)),
            'block_table': torch.stack([s.block_table for s in states]),
            'cos': self.model.cos_buf[flat_pos].unsqueeze(1),
            'sin': self.model.sin_buf[flat_pos].unsqueeze(1),
            'block_size': self.cfg["block_size"]
        }

        input_ids = torch.cat(ids_to_cat)
        return metadata, input_ids

In [5]:
CHOOSE_MODEL = "0.6B"

QWEN3_CONFIG = {
    "max_batch_size":8,
    "block_size":256,
    "vocab_size": 151_936,           # Vocabulary size
    "context_length": 40_960,        # Context length that was used to train the model
    "emb_dim": 1024,                 # Embedding dimension
    "n_heads": 16,                   # Number of attention heads
    "n_layers": 28,                  # Number of layers
    "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
    "head_dim": 128,                 # Size of the heads in GQA
    "qk_norm": True,                 # Whether to normalize queries and keys in GQA
    "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
    "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
    "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
}
torch.manual_seed(123)
device = "cuda" if torch.cuda.is_available() else "cpu"
mgr = PagedKVManager(QWEN3_CONFIG, max_blocks=32, device=device)

# model = Qwen3Model(QWEN3_CONFIG,mgr)
model = Qwen3Model(QWEN3_CONFIG)
model

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [6]:

def load_weights_into_qwen(model, param_config, params):
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")
        
        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))
    
        return left 

    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # QK norms
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # Attention layernorm
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # Final normalization and output head
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

In [7]:
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download



repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}"

local_dir = Path(repo_id).parts[-1]

if CHOOSE_MODEL == "0.6B":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
else:
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

In [8]:
load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device)
del weights_dict

In [9]:
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s

In [10]:
tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}/tokenizer.json"


hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)


tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True)

In [11]:
prompt = "Give me a short introduction to large language models."

input_token_ids1 = tokenizer.encode(prompt)

In [12]:
prompt = "Give me a short introduction to AI"

input_token_ids3 = tokenizer.encode(prompt)

In [13]:
prompt = "Give me a short introduction to Embeddings in nlp"

input_token_ids2 = tokenizer.encode(prompt)

In [14]:
prompt = "Give me a short introduction to python programming"

input_token_ids4 = tokenizer.encode(prompt)

In [15]:
prompt = "How many r's in strawberries?"
input_token_ids5 = tokenizer.encode(prompt)

In [16]:
prompt = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"
input_token_ids6 = tokenizer.encode(prompt)

In [17]:
engine = ContinuousBatchEngine(model, mgr, QWEN3_CONFIG)

In [18]:
my_prompts = [
    input_token_ids1, 
    input_token_ids2,  
    input_token_ids3,
    input_token_ids4,
    input_token_ids5,
    input_token_ids6
]
for prompt in my_prompts:
    engine.add_sequence(prompt,max_gen_len = 512)
engine.active = {}

while engine.waiting_room or engine.active:
    finished = engine.step()
     # Print results as they come in
    for sid, tokens in finished.items():
        print(f"Slot {sid} finished. Total length: {len(tokens)}")
        print(tokenizer.decode(tokens))
        print("-" * 30)

Slot 2 finished. Total length: 196
<|im_start|>user
Give me a short introduction to AI<|im_end|>
<|im_start|>assistant
<think>
Okay, the user wants a short introduction to AI. Let me start by defining what AI is. I should mention that it's a field of study that involves machines learning from data. I need to highlight key aspects like machine learning, natural language processing, and computer vision. Maybe include some examples to make it relatable. Also, touch on the benefits and challenges, like ethical considerations. Keep it concise but informative. Let me check if I'm covering all important points without being too technical. Yeah, that should work.
</think>

AI, or artificial intelligence, refers to the simulation of human intelligence tasks performed by machines. It encompasses technologies like machine learning, natural language processing, and computer vision, enabling systems to learn from data, make decisions, and interact with users in complex ways. AI has applications in 